# 04 — Understand causal attention

**Goal:** calculate how each token gathers information from earlier tokens to form its context.

This follows the sliding windows: windows create training sequences; attention builds a contextual representation for each position.

By the end, you will be able to explain queries (**Q**), keys (**K**), values (**V**), dot products, softmax, and the causal mask. We use small vectors and standard Python to make each calculation visible. No data files are loaded and no model is trained.

## 1. Why context?

In “salam, labas?”, a token's meaning partly depends on the tokens around it. Attention computes a weighted combination of available information for each position.

In GPT, each token is first represented by a vector. The Q, K, and V projection matrices are learned during training. To make the mechanism easy to inspect, our example uses hand-written vectors and matrices; they were not learned from Darija.

In [1]:
tokens = ["salam", "kidayr", "labas", "?"]

# Synthetic teaching vectors: one row per token, three numbers per vector.
embeddings = [
    [1.0, 0.0, 1.0],  # salam
    [0.0, 2.0, 1.0],  # kidayr
    [1.0, 1.0, 0.0],  # labas
    [0.0, 1.0, 2.0],  # ?
]

for token, vector in zip(tokens, embeddings):
    print(f"{token:>7} -> {vector}")

  salam -> [1.0, 0.0, 1.0]
 kidayr -> [0.0, 2.0, 1.0]
  labas -> [1.0, 1.0, 0.0]
      ? -> [0.0, 1.0, 2.0]


## 2. Queries, keys, and values

For each token, three projections produce **Q**, **K**, and **V**. Intuitively, Q describes what a position is looking for, K what each position can signal, and V the information it carries.

We compare Q with every K using a dot product. After division by √dₖ and softmax, the weights sum to 1. We use those weights to compute a weighted average of V. Dividing by √dₖ keeps scores at a reasonable scale as vectors grow.

In [2]:
import math

def linear(vector, matrix):
    """Project a row vector through a matrix stored by columns."""
    return [
        sum(value * matrix[row][column] for row, value in enumerate(vector))
        for column in range(len(matrix[0]))
    ]

def dot(left, right):
    return sum(a * b for a, b in zip(left, right))

def softmax(scores):
    peak = max(scores)
    exponents = [math.exp(score - peak) for score in scores]
    total = sum(exponents)
    return [value / total for value in exponents]

To isolate the calculation, the three matrices below are **fixed example weights** with dimensions 3 by 2. In a real Transformer, these weights are initialized and then updated by gradient descent.

In [3]:
W_q = [[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]]
W_k = [[1.0, 1.0], [1.0, 0.0], [0.0, 1.0]]
W_v = [[1.0, 0.0], [0.0, 1.0], [1.0, -1.0]]

queries = [linear(x, W_q) for x in embeddings]
keys = [linear(x, W_k) for x in embeddings]
values = [linear(x, W_v) for x in embeddings]

print("Q[0] :", queries[0])
print("K[0] :", keys[0])
print("V[0] :", values[0])

Q[0] : [2.0, 1.0]
K[0] : [1.0, 2.0]
V[0] : [2.0, -1.0]


## 3. Block access to the future

During generation, the model must predict the next token without knowing the answer. The **causal mask** prevents position i from consulting positions j > i.

Before softmax, future scores become −∞. Their weights after softmax are zero. The table below shows the attention weights: each row is the position doing the lookup, and each column is a position it can consult.

In [4]:
def causal_attention_weights(queries, keys):
    weights = []
    scale = math.sqrt(len(keys[0]))

    for i, query in enumerate(queries):
        scores = [dot(query, key) / scale for key in keys]
        masked_scores = [score if j <= i else float("-inf")
                         for j, score in enumerate(scores)]
        weights.append(softmax(masked_scores))

    return weights

weights = causal_attention_weights(queries, keys)
for token, row in zip(tokens, weights):
    print(f"{token:>7} -> {[round(weight, 3) for weight in row]}")

assert all(abs(sum(row) - 1.0) < 1e-9 for row in weights)
assert all(weights[i][j] == 0.0 for i in range(len(tokens))
           for j in range(i + 1, len(tokens)))

  salam -> [1.0, 0.0, 0.0, 0.0]
 kidayr -> [0.804, 0.196, 0.0, 0.0]
  labas -> [0.333, 0.333, 0.333, 0.0]
      ? -> [0.335, 0.165, 0.165, 0.335]


## 4. Combine the values

Once the weights are computed, we apply them to the V vectors. The output at each position is its **context vector**: a weighted sum of the accessible values.

We can also check causality by changing the final token. Earlier outputs should remain unchanged because those positions could not see it.

In [5]:
def apply_attention(weights, values):
    outputs = []
    for row in weights:
        output = [
            sum(row[j] * values[j][dimension] for j in range(len(values)))
            for dimension in range(len(values[0]))
        ]
        outputs.append(output)
    return outputs

context_vectors = apply_attention(weights, values)
for token, vector in zip(tokens, context_vectors):
    print(f"{token:>7} -> {[round(value, 3) for value in vector]}")

changed_embeddings = [row[:] for row in embeddings]
changed_embeddings[-1] = [100.0, -50.0, 25.0]
changed_keys = [linear(x, W_k) for x in changed_embeddings]
changed_values = [linear(x, W_v) for x in changed_embeddings]
changed_weights = causal_attention_weights(queries, changed_keys)
changed_context = apply_attention(changed_weights, changed_values)

for i in range(len(tokens) - 1):
    assert all(abs(a - b) < 1e-9
               for a, b in zip(context_vectors[i], changed_context[i]))
print("Past positions do not change when the future changes.")

  salam -> [2.0, -1.0]
 kidayr -> [1.804, -0.609]
  labas -> [1.333, 0.333]
      ? -> [1.67, -0.34]
Past positions do not change when the future changes.


## 5. Key points

- Q and K produce compatibility scores; V carries the information to combine.
- Softmax turns scores into positive weights that sum to 1.
- Division by √dₖ scales scores before softmax.
- The causal mask removes the influence of future tokens.
- All values and matrices here are fixed examples. We do not use the Darija tokenizer or corpus, and no model has been trained.

These loops make the operations visible. The next notebook rewrites the same calculation with PyTorch; multi-head attention and the Transformer block will follow.